# 🏥 Análise de Hábitos de Saúde
**Projeto de Análise de Dados para Iniciantes**

Neste projeto, vamos analisar dados fictícios sobre hábitos de saúde de 40 pessoas.
Vamos responder perguntas como:
- 😴 As pessoas dormem o suficiente?
- 🏃 Quem se exercita mais — homens ou mulheres?
- 💧 A ingestão de água tem relação com o nível de estresse?
- 📊 Qual é o IMC médio dos participantes?

---
**Ferramentas que vamos usar:**
- `pandas` → para ler e manipular os dados (como uma planilha Excel)
- `matplotlib` → para criar gráficos
- `seaborn` → para criar gráficos mais bonitos


## 📦 Etapa 1 — Importar as bibliotecas
Antes de começar, precisamos 'chamar' as ferramentas que vamos usar.
É como abrir o Excel antes de trabalhar com uma planilha.

In [ ]:
# Importamos as bibliotecas (ferramentas) que vamos precisar
import pandas as pd          # 'pd' é um apelido para escrever menos
import matplotlib.pyplot as plt  # 'plt' é o apelido padrão do mercado
import seaborn as sns        # 'sns' também é apelido padrão

# Configurações visuais: deixa os gráficos maiores e mais bonitos
plt.rcParams['figure.figsize'] = (10, 5)
sns.set_theme(style='whitegrid', palette='pastel')

print('✅ Bibliotecas importadas com sucesso!')

## 📂 Etapa 2 — Carregar os dados
Vamos ler o arquivo CSV (que é como uma planilha) e guardar numa variável chamada `df`.
> 💡 `df` vem de **DataFrame** — é o nome que o pandas dá para tabelas de dados.

In [ ]:
# pd.read_csv() lê o arquivo e transforma em tabela
df = pd.read_csv('dados/saude_dados.csv')

# .head(5) mostra as 5 primeiras linhas — útil pra ver se carregou certo
df.head(5)

## 🔍 Etapa 3 — Conhecer os dados
Antes de analisar, precisamos entender o que temos. Isso se chama **Análise Exploratória de Dados (EDA)**.

In [ ]:
# .shape retorna (quantidade de linhas, quantidade de colunas)
linhas, colunas = df.shape
print(f'📊 O dataset tem {linhas} pessoas e {colunas} informações por pessoa.')
print()

# .info() mostra o tipo de cada coluna e se tem valores faltando
print('📋 Informações do dataset:')
df.info()

In [ ]:
# .describe() calcula estatísticas básicas: média, mínimo, máximo, etc.
# Só funciona nas colunas numéricas
df.describe().round(2)

## 🧮 Etapa 4 — Calcular o IMC
O IMC (Índice de Massa Corporal) é uma medida muito usada em saúde.
**Fórmula:** IMC = peso (kg) ÷ altura (m)²

In [ ]:
# Criamos uma nova coluna 'imc' calculada a partir de peso e altura
# Dividimos a altura por 100 pra converter de cm para metros
df['imc'] = df['peso_kg'] / ((df['altura_cm'] / 100) ** 2)
df['imc'] = df['imc'].round(1)  # Arredonda pra 1 casa decimal

# Criamos uma função que classifica o IMC
def classificar_imc(imc):
    if imc < 18.5:
        return 'Abaixo do peso'
    elif imc < 25:
        return 'Peso normal'
    elif imc < 30:
        return 'Sobrepeso'
    else:
        return 'Obesidade'

# .apply() aplica a função em cada linha da coluna 'imc'
df['classificacao_imc'] = df['imc'].apply(classificar_imc)

# Mostra as colunas novas
df[['nome', 'peso_kg', 'altura_cm', 'imc', 'classificacao_imc']].head(8)

## 📊 Etapa 5 — Visualizações (Gráficos)
Agora a parte mais divertida! Vamos criar gráficos para entender os dados visualmente.

In [ ]:
# --- GRÁFICO 1: Distribuição do IMC ---

# Contamos quantas pessoas tem em cada classificação
contagem_imc = df['classificacao_imc'].value_counts()

# Criamos o gráfico de barras
fig, axes = plt.subplots(1, 2, figsize=(14, 5))  # 2 gráficos lado a lado

# Gráfico de barras (esquerda)
contagem_imc.plot(kind='bar', ax=axes[0], color=['#4CAF50','#FFC107','#FF9800','#F44336'],
                  edgecolor='white', linewidth=1.5)
axes[0].set_title('Classificação do IMC dos Participantes', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Classificação')
axes[0].set_ylabel('Quantidade de Pessoas')
axes[0].tick_params(axis='x', rotation=15)

# Gráfico de pizza (direita)
axes[1].pie(contagem_imc.values, labels=contagem_imc.index,
            autopct='%1.1f%%', colors=['#4CAF50','#FFC107','#FF9800','#F44336'],
            startangle=90)
axes[1].set_title('Proporção do IMC', fontsize=14, fontweight='bold')

plt.tight_layout()  # Ajusta o espaçamento entre gráficos
plt.savefig('grafico_imc.png', dpi=150, bbox_inches='tight')  # Salva a imagem
plt.show()
print(f'\n📌 IMC médio: {df["imc"].mean():.1f}')

In [ ]:
# --- GRÁFICO 2: Horas de Sono ---

# A OMS recomenda entre 7 e 9 horas de sono para adultos
sono_ok = df[df['horas_sono'].between(7, 9)].shape[0]   # .between() checa o intervalo
sono_total = df.shape[0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma: mostra a distribuição das horas de sono
axes[0].hist(df['horas_sono'], bins=10, color='#5C85D6', edgecolor='white', linewidth=1.5)
axes[0].axvline(7, color='green', linestyle='--', label='Mínimo OMS (7h)')
axes[0].axvline(9, color='red', linestyle='--', label='Máximo OMS (9h)')
axes[0].set_title('Distribuição das Horas de Sono', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Horas de Sono')
axes[0].set_ylabel('Quantidade de Pessoas')
axes[0].legend()

# Comparação: Dentro vs Fora da recomendação
categorias = ['Dentro da\nrecomendação (7-9h)', 'Fora da\nrecomendação']
valores = [sono_ok, sono_total - sono_ok]
axes[1].bar(categorias, valores, color=['#4CAF50', '#F44336'], edgecolor='white', linewidth=1.5)
axes[1].set_title('Sono: Dentro da Recomendação da OMS?', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Quantidade de Pessoas')

plt.tight_layout()
plt.savefig('grafico_sono.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\n😴 Média de sono: {df["horas_sono"].mean():.1f}h')
print(f'✅ {sono_ok} de {sono_total} pessoas dormem dentro da recomendação da OMS.')

In [ ]:
# --- GRÁFICO 3: Exercício por Sexo ---

# .groupby() agrupa os dados por categoria
# .mean() calcula a média de cada grupo
exercicio_por_sexo = df.groupby('sexo')['dias_exercicio_semana'].mean().round(1)

fig, ax = plt.subplots(figsize=(8, 5))
barras = ax.bar(['Feminino (F)', 'Masculino (M)'],
                exercicio_por_sexo.values,
                color=['#E91E8C', '#2196F3'], edgecolor='white', linewidth=1.5,
                width=0.5)

# Adiciona o número em cima de cada barra
for barra in barras:
    altura = barra.get_height()
    ax.text(barra.get_x() + barra.get_width()/2., altura + 0.05,
            f'{altura} dias/sem', ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_title('Média de Dias de Exercício por Semana (por Sexo)', fontsize=14, fontweight='bold')
ax.set_ylabel('Dias por Semana')
ax.set_ylim(0, 6)

plt.tight_layout()
plt.savefig('grafico_exercicio.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- GRÁFICO 4: Água vs Estresse ---
# Existe relação entre beber mais água e ter menos estresse?

fig, ax = plt.subplots(figsize=(9, 5))

# Scatter plot: cada ponto é uma pessoa
scatter = ax.scatter(df['litros_agua_dia'], df['nivel_estresse'],
                     c=df['nivel_estresse'], cmap='RdYlGn_r',  # verde=baixo, vermelho=alto
                     s=100, alpha=0.7, edgecolors='white')

plt.colorbar(scatter, ax=ax, label='Nível de Estresse')
ax.set_title('Consumo de Água vs Nível de Estresse', fontsize=14, fontweight='bold')
ax.set_xlabel('Litros de Água por Dia')
ax.set_ylabel('Nível de Estresse (1-10)')

plt.tight_layout()
plt.savefig('grafico_agua_estresse.png', dpi=150, bbox_inches='tight')
plt.show()

# .corr() calcula a correlação entre duas colunas
# Varia de -1 a 1: perto de -1 = relação inversa, perto de 0 = sem relação, perto de 1 = relação direta
correlacao = df['litros_agua_dia'].corr(df['nivel_estresse']).round(3)
print(f'🔗 Correlação entre água e estresse: {correlacao}')
print('(Valores próximos de 0 indicam pouca relação linear entre as variáveis)')

## 📝 Etapa 6 — Conclusões
Após a análise, vamos resumir o que descobrimos!

In [ ]:
print('=' * 55)
print('       📊 RESUMO DA ANÁLISE DE SAÚDE')
print('=' * 55)
print(f'👥 Total de participantes analisados: {df.shape[0]}')
print()
print('📏 IMC')
print(f'   Média geral: {df["imc"].mean():.1f}')
for classif, qtd in df['classificacao_imc'].value_counts().items():
    print(f'   {classif}: {qtd} pessoas')
print()
print('😴 SONO')
print(f'   Média: {df["horas_sono"].mean():.1f}h por noite')
print(f'   Dentro da recomendação OMS (7-9h): {df["horas_sono"].between(7,9).sum()} pessoas')
print()
print('🏃 EXERCÍCIO')
for sexo, media in df.groupby('sexo')['dias_exercicio_semana'].mean().items():
    nome_sexo = 'Feminino' if sexo == 'F' else 'Masculino'
    print(f'   {nome_sexo}: {media:.1f} dias/semana em média')
print()
print('💧 ÁGUA')
print(f'   Média de consumo: {df["litros_agua_dia"].mean():.1f}L/dia')
print(f'   (OMS recomenda pelo menos 2L/dia)')
print()
print('🚬 TABAGISMO')
for resp, qtd in df['fuma'].value_counts().items():
    print(f'   Fuma {resp}: {qtd} pessoas')
print('=' * 55)